# PyNRPF v0.1.0  Hyperparameter Search

Runs the publication-only DTR sweep and XGBoost random search, then exports CSV summaries to `outputs/publication_tables/`.


In [1]:
#  Environment + imports
from pathlib import Path
import hashlib
import sys, random

# Make src importable
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.hyperparameter_search import run_m7_one_at_a_time_sweep, run_m8_random_search
from src.io import (
    load_yaml, req, get,
    verify_sha256_best_effort, load_parquet,
    ensure_dir,
)
from src.validate import basic_validate

print("Python:", sys.version)
print("CWD:   ", Path.cwd())
print("REPO:  ", REPO_ROOT)


Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
CWD:    C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\notebooks
REPO:   C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper


In [2]:
#  CONFIG
CFG_PATH = REPO_ROOT / "config" / "run.yaml"
cfg = load_yaml(CFG_PATH)
print("Config loaded from:", CFG_PATH)

RUN_TAG = str(req(cfg, "run.run_tag"))
SEED = int(req(cfg, "run.seed"))
SEARCH_SEED = 123
random.seed(SEED)
np.random.seed(SEED)

DATASET_PATH = (REPO_ROOT / str(req(cfg, "paths.dataset_parquet"))).resolve()
SHA_PATH     = (REPO_ROOT / str(req(cfg, "paths.sha256_file"))).resolve()
OUTPUT_DIR   = (REPO_ROOT / str(req(cfg, "paths.output_dir"))).resolve()
TABLE_DIR    = OUTPUT_DIR / "publication_tables"
ensure_dir(TABLE_DIR)

M7_SEARCH_PATH = TABLE_DIR / "m7_dtr_hyperparameter_sweep.csv"
M8_SEARCH_PATH = TABLE_DIR / "m8_xgb_random_search.csv"
XGB1_PATH = OUTPUT_DIR / "xgb1_day.pkl"
XGB2_PATH = OUTPUT_DIR / "xgb2_timestamp.pkl"

COL_SITE  = str(req(cfg, "data.columns.site"))
COL_TS    = str(req(cfg, "data.columns.ts"))
COL_NET   = str(req(cfg, "data.columns.net_load"))
COL_SOLAR = str(req(cfg, "data.columns.solar"))
COL_GT    = str(req(cfg, "data.columns.gt"))
ALL_COLS  = [COL_SITE, COL_TS, COL_NET, COL_SOLAR, COL_GT]
INTERVAL_MINUTES = int(req(cfg, "data.interval_minutes"))

VERIFY_SHA256           = bool(get(cfg, "validation.verify_sha256_best_effort", True))
STRIP_TIMEZONE          = bool(get(cfg, "validation.strip_timezone", True))
ENFORCE_INTERVAL_ALIGN  = bool(get(cfg, "validation.enforce_interval_alignment", True))
ENFORCE_UNIQUE_KEYS     = bool(get(cfg, "validation.enforce_unique_keys", True))

print(f"RUN_TAG:        {RUN_TAG}")
print(f"SEED:           {SEED}")
print(f"SEARCH_SEED:    {SEARCH_SEED}")
print(f"DATASET:        {DATASET_PATH}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")
print(f"TABLE_DIR:      {TABLE_DIR}")
print(f"M7_SEARCH_CSV:  {M7_SEARCH_PATH}")
print(f"M8_SEARCH_CSV:  {M8_SEARCH_PATH}")


Config loaded from: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\config\run.yaml
RUN_TAG:        local_dev
SEED:           123
SEARCH_SEED:    123
DATASET:        C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\dataset\raw\rpf_dataset.parquet
OUTPUT_DIR:     C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs
TABLE_DIR:      C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables
M7_SEARCH_CSV:  C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\m7_dtr_hyperparameter_sweep.csv
M8_SEARCH_CSV:  C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\m8_xgb_random_search.csv


In [3]:
#  Ensure required inputs exist
if not DATASET_PATH.exists():
    print("Dataset not found locally.")
    print("Please place the parquet at:", DATASET_PATH)
    raise SystemExit("Stopping: no local dataset available.")

for model_path in [XGB1_PATH, XGB2_PATH]:
    if not model_path.exists():
        raise FileNotFoundError(f"Baseline model artifact not found: {model_path}")

local_path = DATASET_PATH
print("Parquet found locally:", local_path)
print("Baseline models found:")
print(f"  - {XGB1_PATH}")
print(f"  - {XGB2_PATH}")


Parquet found locally:

 C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\dataset\raw\rpf_dataset.parquet
Baseline models found:
  - C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\xgb1_day.pkl
  - C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\xgb2_timestamp.pkl


In [4]:
#  Load dataset
if VERIFY_SHA256:
    sha_result = verify_sha256_best_effort(local_path, SHA_PATH)
    print("SHA-256 check:", sha_result["status"],
          f"({sha_result.get('note', '')})" if sha_result.get("note") else "")

df = load_parquet(local_path)
print(df.dtypes)
df.head()


SHA-256 check: ok 


Loaded 1,011,264 rows x 5 cols from rpf_dataset.parquet
substation_id                         object
timestamp                datetime64[ns, UTC]
net_load_MW                          float64
solar_MW                             float64
net_load_ground_truth                float64
dtype: object


,substation_id,timestamp,net_load_MW,solar_MW,net_load_ground_truth
0,A,2021-11-01 00:15:00+00:00,NaN,0.0,NaN
1,A,2021-11-01 00:30:00+00:00,NaN,0.0,NaN
2,A,2021-11-01 00:45:00+00:00,NaN,0.0,NaN
3,A,2021-11-01 01:00:00+00:00,NaN,0.0,NaN
4,A,2021-11-01 01:15:00+00:00,12.515586,0.0,12.515586


In [5]:
#  Validation
result = basic_validate(
    df,
    cols_required=ALL_COLS,
    site_col=COL_SITE,
    ts_col=COL_TS,
    key_cols=[COL_SITE, COL_TS],
    interval_minutes=INTERVAL_MINUTES,
    strip_timezone=STRIP_TIMEZONE,
    enforce_interval_alignment=ENFORCE_INTERVAL_ALIGN,
    enforce_unique_keys=ENFORCE_UNIQUE_KEYS,
)

df = result["df"]
summary = result["summary"]

print("Validation passed.")
for k, v in summary.items():
    print(f"  {k}: {v}")


Validation passed.
  n_rows: 1011264
  n_sites: 10
  n_duplicate_keys: 0
  min_ts: 2021-11-01 00:15:00
  max_ts: 2024-09-30 00:00:00
  null_substation_id: 0
  null_timestamp: 0
  null_net_load_MW: 3000
  null_solar_MW: 1040
  null_net_load_ground_truth: 3000


In [6]:
#  Baseline artifact hashes
def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

baseline_hashes = {
    XGB1_PATH.name: _sha256(XGB1_PATH),
    XGB2_PATH.name: _sha256(XGB2_PATH),
}
baseline_hashes


{'xgb1_day.pkl': 'f4dadc9ecd270565388264501e7389a1d77c9d2b2dc52569055e217eceabfd92',
 'xgb2_timestamp.pkl': '9a766fc769ee7691f27f803ccb5525f9edee1c39424b3b14e75a5c50c74d9020'}

In [7]:
#  Run sweeps and export CSVs
m7_search = run_m7_one_at_a_time_sweep(
    df,
    cfg,
    COL_SITE,
    COL_TS,
    COL_NET,
    COL_SOLAR,
    COL_GT,
)

m8_search = run_m8_random_search(
    df,
    cfg,
    COL_SITE,
    COL_TS,
    COL_NET,
    COL_SOLAR,
    COL_GT,
    seed=SEARCH_SEED,
    trials=5,
)

m7_search.to_csv(M7_SEARCH_PATH, index=False)
m8_search.to_csv(M8_SEARCH_PATH, index=False)

expected_m7_cols = [
    "sweep_name",
    "solar_peak_window_hours",
    "min_threshold",
    "min_threshold_both",
    "day_precision",
    "day_recall",
    "day_f1",
    "interval_tp_precision",
    "interval_tp_recall",
    "interval_tp_f1",
]
expected_m8_cols = [
    "eta",
    "max_depth",
    "scale_pos_weight",
    "day_precision",
    "day_recall",
    "day_f1",
    "interval_tp_precision",
    "interval_tp_recall",
    "interval_tp_f1",
]
assert list(m7_search.columns) == expected_m7_cols, "Unexpected m7 search CSV columns"
assert list(m8_search.columns) == expected_m8_cols, "Unexpected m8 search CSV columns"
assert len(m7_search) == 9, f"Expected 9 m7 sweep rows, found {len(m7_search)}"
assert len(m8_search) == 5, f"Expected 5 m8 search rows, found {len(m8_search)}"
assert len(m8_search[["eta", "max_depth", "scale_pos_weight"]].drop_duplicates()) == 5, "m8 hyperparameter triplets must be unique"

m8_cfg = req(cfg, "m8_xgb")
xgb1_cfg = req(m8_cfg, "xgb1_day")
default_triplet = (
    round(float(xgb1_cfg["eta"]), 4),
    int(xgb1_cfg["max_depth"]),
    round(float(xgb1_cfg["scale_pos_weight"]), 3),
)
actual_triplet = tuple(m8_search.loc[0, ["eta", "max_depth", "scale_pos_weight"]].tolist())
assert actual_triplet == default_triplet, f"First m8 trial should be baseline config, got {actual_triplet}"

after_hashes = {
    XGB1_PATH.name: _sha256(XGB1_PATH),
    XGB2_PATH.name: _sha256(XGB2_PATH),
}
assert after_hashes == baseline_hashes, "Baseline m8 model artifacts changed during search execution"

print(f"Wrote: {M7_SEARCH_PATH}")
print(f"Wrote: {M8_SEARCH_PATH}")
print("Baseline model artifact hashes unchanged.")



m7_threshold complete (19.5s):
  Site-days total:           10,643
  Skipped (missing data):    90
  Skipped (negative MW):     0
  Skipped (midday < 3 pts):  109
  Skipped (no candidates):   392
  Skipped (no valid pair):   5,470
  Skipped (threshold gate):  939
  Skipped (daytime gate):    3
  RPF days flagged:          3,640
  Intervals flagged:         45,775
  Thresholds: min=5.00%, both=25.00%



m7_threshold complete (22.8s):
  Site-days total:           10,643
  Skipped (missing data):    90
  Skipped (negative MW):     0
  Skipped (midday < 3 pts):  109
  Skipped (no candidates):   191
  Skipped (no valid pair):   5,514
  Skipped (threshold gate):  1,022
  Skipped (daytime gate):    3
  RPF days flagged:          3,714
  Intervals flagged:         44,769
  Thresholds: min=5.00%, both=25.00%



m7_threshold complete (8.4s):
  Site-days total:           10,643
  Skipped (missing data):    90
  Skipped (negative MW):     0
  Skipped (midday < 3 pts):  109
  Skipped (no candidates):   141
  Skipped (no valid pair):   5,520
  Skipped (threshold gate):  1,059
  Skipped (daytime gate):    3
  RPF days flagged:          3,721
  Intervals flagged:         43,506
  Thresholds: min=5.00%, both=25.00%



m7_threshold complete (4.0s):
  Site-days total:           10,643
  Skipped (missing data):    90
  Skipped (negative MW):     0
  Skipped (midday < 3 pts):  109
  Skipped (no candidates):   191
  Skipped (no valid pair):   5,514
  Skipped (threshold gate):  2,649
  Skipped (daytime gate):    1
  RPF days flagged:          2,089
  Intervals flagged:         24,379
  Thresholds: min=1.00%, both=25.00%



m7_threshold complete (4.4s):
  Site-days total:           10,643
  Skipped (missing data):    90
  Skipped (negative MW):     0
  Skipped (midday < 3 pts):  109
  Skipped (no candidates):   191
  Skipped (no valid pair):   5,514
  Skipped (threshold gate):  1,542
  Skipped (daytime gate):    1
  RPF days flagged:          3,196
  Intervals flagged:         39,217
  Thresholds: min=2.50%, both=25.00%


...

XGB2 features: 665 columns, 177,528 rows (47,705 positive)


C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\src\hyperparameter_search.py:182: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["m8_rpf_flag"].fillna(False).infer_objects(copy=False).astype(bool)


XGB2 features: 665 columns, 177,944 rows (47,724 positive)


C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\src\hyperparameter_search.py:182: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["m8_rpf_flag"].fillna(False).infer_objects(copy=False).astype(bool)


Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\m7_dtr_hyperparameter_sweep.csv
Wrote: C:\Users\z5404477\Documents\PyNRPF\publication\1_conference_paper\outputs\publication_tables\m8_xgb_random_search.csv
Baseline model artifact hashes unchanged.


In [8]:
#  m7_dtr one-at-a-time sweep results
m7_search


,sweep_name,solar_peak_window_hours,min_threshold,min_threshold_both,day_precision,day_recall,day_f1,interval_tp_precision,interval_tp_recall,interval_tp_f1
0,solar_peak_window_hours,1.5,0.050,0.25,0.915,0.943,0.929,0.865,0.823,0.843
1,solar_peak_window_hours,2.5,0.050,0.25,0.912,0.963,0.937,0.869,0.796,0.831
2,solar_peak_window_hours,3.5,0.050,0.25,0.911,0.965,0.937,0.871,0.774,0.820
3,min_threshold,2.5,0.010,0.25,0.965,0.535,0.689,0.855,0.769,0.810
4,min_threshold,2.5,0.025,0.25,0.942,0.852,0.894,0.871,0.793,0.830
5,min_threshold,2.5,0.040,0.25,0.923,0.943,0.933,0.869,0.794,0.830
6,min_threshold_both,2.5,0.050,0.15,0.925,0.951,0.938,0.872,0.795,0.832
7,min_threshold_both,2.5,0.050,0.25,0.912,0.963,0.937,0.869,0.796,0.831
8,min_threshold_both,2.5,0.050,0.35,0.908,0.965,0.936,0.868,0.796,0.830


In [9]:
#  m8_xgb random-search results
m8_search


,eta,max_depth,scale_pos_weight,day_precision,day_recall,day_f1,interval_tp_precision,interval_tp_recall,interval_tp_f1
0,0.1000,6,5.000,0.954,0.958,0.956,0.928,0.796,0.857
1,0.1444,8,2.851,0.958,0.953,0.955,0.948,0.763,0.845
2,0.0459,4,2.655,0.962,0.956,0.959,0.979,0.337,0.501
3,0.1946,7,3.121,0.960,0.953,0.957,0.945,0.765,0.845
4,0.1981,10,8.376,0.957,0.956,0.957,0.927,0.826,0.873
